# 06 Prototype (Prototyp) | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: kopiowanie bez zaleznosci od klas
2. 📋 Plytka vs gleboka kopia
3. 🔧 `__copy__` i `__deepcopy__`
4. 📦 Rejestr prototypow
5. 🌍 Zastosowania i kiedy uzywac

## 1. 🔹 Problem: kopiowanie bez zaleznosci od klas

Prototype to wzorzec kreacyjny pozwalajacy kopiowac istniejace
obiekty bez uzalezniania kodu od ich klas.

Problem: chcemy skopiowac obiekt, ale:
- Nie znamy konkretnej klasy (wiemy tylko o interfejsie)
- Obiekt moze miec prywatne pola niedostepne z zewnatrz
- Tworzenie obiektu od zera jest drogie

Rozwiazanie: delegujemy kopiowanie do samego obiektu.
Protokol Pythona: `copy.copy()` wywoluje `__copy__()`,
`copy.deepcopy()` wywoluje `__deepcopy__()`.

> 💡 Prototype rozwiazuje problem kopiowania - nie tworzenia.
> Uzywamy go gdy mamy juz obiekt i chcemy jego kopie.

In [ ]:
import copy

# Problem: kopiowanie bez znajomosci klasy
class Shape:
    def __init__(self, x: int, y: int, color: str):
        self.x = x
        self.y = y
        self.color = color

    def clone(self) -> 'Shape':
        return copy.copy(self)  # deleguje do obiektu!

class Circle(Shape):
    def __init__(self, x: int, y: int, color: str, radius: float):
        super().__init__(x, y, color)
        self.radius = radius

    def __repr__(self) -> str:
        return f'Circle(x={self.x}, y={self.y}, r={self.radius}, {self.color})'

original = Circle(10, 20, 'red', 5.0)
clone = original.clone()
clone.color = 'blue'
clone.x = 30

print(f'Original: {original}')
print(f'Clone:    {clone}')

---

### 🐍 Cwiczenia - problem kopiowania

1. Stworz dwie listy `a = [1, 2, [3, 4]]` i `b = a`. Zmien
   `b[2].append(5)`. Sprawdz `a` - dlaczego sie zmienila?
2. Napisz klase `Config(settings: dict)` i porownaj wynik
   `copy.copy()` vs przypisania (`=`) po zmianie `settings['key']`.
3. *(Trudniejsze)* Napisz funkcje `safe_clone(obj)` ktora
   robi deepcopy jesli obiekt ma atrybut `_mutable`,
   a shallow copy w przeciwnym wypadku.

In [ ]:
# Cwiczenie 1: aliasing listy
a = [1, 2, [3, 4]]
b = a
b[2].append(5)
print(f'a = {a}')  # [1, 2, [3, 4, 5]] - ta sama lista!
print(f'b = {b}')

In [ ]:
# Cwiczenie 2: copy.copy vs przypisanie
import copy

class Config:
    def __init__(self, settings: dict):
        self.settings = settings

cfg = Config({'debug': False, 'db': {'host': 'localhost'}})
alias = cfg
shallow = copy.copy(cfg)

alias.settings['debug'] = True
print(f'cfg.debug via alias: {cfg.settings["debug"]}')  # True (ten sam obiekt)

shallow.settings['db']['host'] = 'prod'
print(f'cfg.db.host via shallow: {cfg.settings["db"]["host"]}')  # prod (wspoldzielony dict)

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: safe_clone
import copy

def safe_clone(obj):
    # hint: hasattr(obj, '_mutable')
    ...

class ImmutableData:
    def __init__(self, value): self.value = value

class MutableData:
    _mutable = True
    def __init__(self, items): self.items = items

immutable = ImmutableData(42)
mutable = MutableData([1, 2, 3])

print(type(safe_clone(immutable)).__name__)
print(type(safe_clone(mutable)).__name__)

## 2. 🔹 Plytka vs gleboka kopia

Modul `copy` oferuje dwa rodzaje kopii:

**Plytka kopia (shallow copy)** - `copy.copy()`:
- Tworzy nowy obiekt najwyzszego poziomu
- Zagniezdzone obiekty sa WSPOLDZIELONE (te same referencje)
- Szybsza, zuzuwa mniej pamieci

**Gleboka kopia (deep copy)** - `copy.deepcopy()`:
- Tworzy nowy obiekt i rekurencyjnie kopiuje wszystkie zagniezdzone
- Wszystkie obiekty sa niezalezne
- Wolniejsza, zuzuwa wiecej pamieci

| | Shallow | Deep |
|---|---|---|
| Nowy obiekt | tak | tak |
| Zagniezdzone obiekty | wspoldzielone | skopiowane |
| Szybkosc | szybsza | wolniejsza |

> ⚠️ Plytka kopia mutowalnych obiektow (list, dict) moze
> prowadzic do nieoczekiwanych efektow ubocznych.

In [ ]:
import copy

original = {
    'name': 'Alice',
    'scores': [95, 87, 92],
    'address': {'city': 'Warsaw', 'zip': '00-001'}
}

shallow = copy.copy(original)
deep = copy.deepcopy(original)

# Zmiana wartosci prostej
shallow['name'] = 'Bob'
deep['name'] = 'Carol'
print(f'After name change:')
print(f'  original: {original["name"]}')  # Alice
print(f'  shallow:  {shallow["name"]}')   # Bob
print(f'  deep:     {deep["name"]}')      # Carol

# Zmiana zagniezdzonej listy
shallow['scores'].append(100)
print(f'After scores append:')
print(f'  original: {original["scores"]}')  # [95, 87, 92, 100] - zmodyfikowane!
print(f'  shallow:  {shallow["scores"]}')   # [95, 87, 92, 100]

deep['scores'].append(200)
print(f'After deep scores append:')
print(f'  original: {original["scores"]}')  # bez zmian
print(f'  deep:     {deep["scores"]}')      # [95, 87, 92, 100, 200]

---

### 🐍 Cwiczenia - shallow vs deep

1. Stworz zagniezdzona strukture: `data = {'users': [{'name': 'Alice'}]}`.
   Zrob shallow copy i zmien `data['users'][0]['name']`. Sprawdz oryginalna.
2. Zrob deep copy tej samej struktury i zmien zagniezdzone dane.
   Sprawdz ze oryginalna nie jest zmieniona.
3. *(Trudniejsze)* Zmierz czas kopiowania duzej struktury
   (np. lista 10000 slownikow z lista 100 elementow).
   Porownaj shallow vs deep uzywajac `time.perf_counter`.

In [ ]:
# Cwiczenie 1: shallow copy ze zmiana zagniezdzonej
import copy
data = {'users': [{'name': 'Alice', 'scores': [10, 20]}]}
shallow = copy.copy(data)
shallow['users'][0]['name'] = 'Bob'  # zmienia oryginalna!
print(f'original: {data}')
print(f'shallow:  {shallow}')

In [ ]:
# Cwiczenie 2: deep copy - niezalezne
import copy
data = {'users': [{'name': 'Alice', 'scores': [10, 20]}]}
deep = copy.deepcopy(data)
deep['users'][0]['name'] = 'Carol'
deep['users'][0]['scores'].append(30)
print(f'original: {data}')  # bez zmian
print(f'deep:     {deep}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: benchmark shallow vs deep
import copy, time

big_data = [{'id': i, 'values': list(range(100))} for i in range(10000)]

start = time.perf_counter()
shallow = copy.copy(big_data)
shallow_time = time.perf_counter() - start

start = time.perf_counter()
deep = copy.deepcopy(big_data)
deep_time = time.perf_counter() - start

print(f'Shallow: {shallow_time:.4f}s')
print(f'Deep:    {deep_time:.4f}s')
print(f'Deep jest {deep_time/shallow_time:.1f}x wolniejsze')

## 3. 🔹 `__copy__` i `__deepcopy__`

Mozemy kontrolowac zachowanie kopiowania przez zdefiniowanie
specjalnych metod:

- `__copy__(self)` - wywolywana przez `copy.copy(obj)`
- `__deepcopy__(self, memo: dict)` - wywolywana przez `copy.deepcopy(obj)`

Po co `memo` w `__deepcopy__`? Slownik `memo` sluzy do zapobiegania
nieskonczonej rekurencji przy kopiowaniu cyklicznych struktur danych.
Przekazujemy `memo` do rekurencyjnych wywolan `copy.deepcopy()`.

> 💡 Sygnatury:
> - `def __copy__(self) -> 'ClassName':`
> - `def __deepcopy__(self, memo: dict) -> 'ClassName':`

In [ ]:
import copy
from typing import Optional

class LinkedNode:
    def __init__(self, value: int, next_node: Optional['LinkedNode'] = None):
        self.value = value
        self.next = next_node

    def __copy__(self) -> 'LinkedNode':
        # plytka kopia: ten sam next (wspoldzielony)
        return LinkedNode(self.value, self.next)

    def __deepcopy__(self, memo: dict) -> 'LinkedNode':
        # Zapobiegamy wielokrotnemu kopiowaniu tego samego obiektu
        if id(self) in memo:
            return memo[id(self)]
        clone = LinkedNode(self.value)
        memo[id(self)] = clone  # zapisz PRZED rekurencja!
        if self.next is not None:
            clone.next = copy.deepcopy(self.next, memo)
        return clone

    def to_list(self) -> list:
        result, current = [], self
        while current:
            result.append(current.value)
            current = current.next
        return result

# Lista: 1 -> 2 -> 3
node3 = LinkedNode(3)
node2 = LinkedNode(2, node3)
node1 = LinkedNode(1, node2)

shallow = copy.copy(node1)
deep = copy.deepcopy(node1)

node2.value = 99  # zmien oryginalna
print(f'Original: {node1.to_list()}')    # [1, 99, 3]
print(f'Shallow:  {shallow.to_list()}')  # [1, 99, 3] - wspoldzielony
print(f'Deep:     {deep.to_list()}')     # [1, 2, 3] - niezalezna kopia

---

### 🐍 Cwiczenia - `__copy__` i `__deepcopy__`

1. Napisz klase `GameState(level, score, inventory: list)`. Zaimplementuj
   `__copy__` kopiujacy plytko i `__deepcopy__` kopiujacy gleboko.
2. Sprawdz ze `copy.copy(state).inventory is state.inventory` to `True`,
   a `copy.deepcopy(state).inventory is state.inventory` to `False`.
3. *(Trudniejsze)* Napisz klase `TreeNode(value, children: list)`.
   Zaimplementuj `__deepcopy__` zapobiegajacy nieskonczonej rekurencji.

In [ ]:
# Cwiczenie 1: GameState z __copy__ i __deepcopy__
import copy

class GameState:
    def __init__(self, level: int, score: int, inventory: list[str]):
        self.level = level
        self.score = score
        self.inventory = inventory

    def __copy__(self) -> 'GameState': ...
    def __deepcopy__(self, memo: dict) -> 'GameState': ...
    def __repr__(self) -> str:
        return f'GameState(lvl={self.level}, score={self.score}, inv={self.inventory})'

state = GameState(5, 1000, ['sword', 'shield'])
shallow = copy.copy(state)
deep = copy.deepcopy(state)
print(state, shallow, deep)

In [ ]:
# Cwiczenie 2: weryfikacja wspoldzielenia
import copy

state = GameState(1, 0, ['bow'])
shallow = copy.copy(state)
deep = copy.deepcopy(state)

print(f'shallow.inventory is state.inventory: {shallow.inventory is state.inventory}')  # True
print(f'deep.inventory is state.inventory: {deep.inventory is state.inventory}')        # False

state.inventory.append('arrows')
print(f'shallow after append: {shallow.inventory}')  # zawiera arrows
print(f'deep after append:    {deep.inventory}')     # nie zawiera arrows

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: TreeNode z __deepcopy__
import copy
from typing import Optional

class TreeNode:
    def __init__(self, value: str):
        self.value = value
        self.children: list['TreeNode'] = []

    def add_child(self, node: 'TreeNode') -> None:
        self.children.append(node)

    def __deepcopy__(self, memo: dict) -> 'TreeNode':
        # hint: zapisz w memo[id(self)] przed rekurencja
        ...

    def to_dict(self) -> dict:
        return {'value': self.value, 'children': [c.to_dict() for c in self.children]}

root = TreeNode('root')
child1 = TreeNode('child1')
child2 = TreeNode('child2')
root.add_child(child1)
root.add_child(child2)
child1.add_child(TreeNode('leaf1'))

clone = copy.deepcopy(root)
clone.value = 'clone_root'
clone.children[0].value = 'clone_child1'

print(f'Original: {root.to_dict()}')
print(f'Clone:    {clone.to_dict()}')

## 4. 🔹 Rejestr prototypow

Rejestr prototypow (Prototype Registry) to slownik przechowujacy
gotowe instancje - szablony - ktore mozna klonowac na zadanie.

Zalety:
- Klient nie musi znac klas produktow
- Centralne zarzadzanie szablonami
- Latwa rejestracja nowych prototypow

Dziala podobnie do rejestru fabryk (Factory Method), ale zamiast
klas lub funkcji przechowuje gotowe instancje do sklonowania.

> 💡 Rejestr prototypow jest szczegolnie uzyteczny gdy:
> - Obiekty sa tworzone z konfiguracji
> - Tworzenie od zera jest drogie
> - Mamy kilka wariantow obiektu (templates)

In [ ]:
import copy
from dataclasses import dataclass, field

@dataclass
class DocumentTemplate:
    title: str
    font: str = 'Arial'
    font_size: int = 12
    margins: dict = field(default_factory=lambda: {'top': 2.5, 'bottom': 2.5})
    content: list[str] = field(default_factory=list)

    def clone(self) -> 'DocumentTemplate':
        return copy.deepcopy(self)

class TemplateRegistry:
    def __init__(self):
        self._templates: dict[str, DocumentTemplate] = {}

    def register(self, key: str, template: DocumentTemplate) -> None:
        self._templates[key] = template

    def clone(self, key: str) -> DocumentTemplate:
        if key not in self._templates:
            raise KeyError(f'Unknown template: {key}')
        return self._templates[key].clone()

    def list_templates(self) -> list[str]:
        return list(self._templates.keys())

# Tworzymy szablony raz
registry = TemplateRegistry()
registry.register('report', DocumentTemplate(
    title='Monthly Report',
    font='Times New Roman',
    font_size=11,
    margins={'top': 2.0, 'bottom': 2.0},
))
registry.register('letter', DocumentTemplate(
    title='Business Letter',
    font='Arial',
    font_size=12,
))
registry.register('invoice', DocumentTemplate(
    title='Invoice',
    font='Calibri',
    font_size=10,
))

print('Available templates:', registry.list_templates())

# Klonujemy i dostosowujemy
my_report = registry.clone('report')
my_report.title = 'Q1 2024 Sales Report'
my_report.content.append('Introduction')

another_report = registry.clone('report')  # swiezy klon, bez zmian
print(f'my_report:      {my_report.title}')
print(f'another_report: {another_report.title}')  # Monthly Report

---

### 🐍 Cwiczenia - rejestr prototypow

1. Napisz `CharacterRegistry` dla gry RPG przechowujacy szablony
   postaci: warrior, mage, archer z domyslnymi statystykami.
2. Sklonuj dwie postacie wojownikow z rejestru i nadaj im
   rozne imiona. Sprawdz ze statystyki sa niezalezne.
3. *(Trudniejsze)* Rozszerz rejestr o metode `register_from_dict(key, data)`
   tworzaca prototyp z slownika konfiguracyjnego.

In [ ]:
# Cwiczenie 1: CharacterRegistry
import copy
from dataclasses import dataclass, field

@dataclass
class Character:
    name: str
    char_class: str
    hp: int
    mp: int
    stats: dict = field(default_factory=dict)
    inventory: list[str] = field(default_factory=list)

    def clone(self, new_name: str) -> 'Character':
        ...

class CharacterRegistry:
    def __init__(self): self._templates: dict = {}
    def register(self, key: str, char: Character) -> None: ...
    def create(self, key: str, name: str) -> Character: ...

registry = CharacterRegistry()
registry.register('warrior', Character('Template', 'warrior', 100, 20,
    {'str': 18, 'dex': 12}, ['sword']))
registry.register('mage', Character('Template', 'mage', 60, 100,
    {'int': 20, 'wis': 18}, ['staff']))

print(registry.create('warrior', 'Arthur'))
print(registry.create('mage', 'Merlin'))

In [ ]:
# Cwiczenie 2: niezaleznosc klonow
w1 = registry.create('warrior', 'Lancelot')
w2 = registry.create('warrior', 'Percival')

w1.stats['str'] = 20
w1.inventory.append('shield')

print(f'w1: {w1.name}, str={w1.stats["str"]}, inv={w1.inventory}')
print(f'w2: {w2.name}, str={w2.stats["str"]}, inv={w2.inventory}')  # bez zmian

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: register_from_dict
class ExtendedCharacterRegistry(CharacterRegistry):
    def register_from_dict(self, key: str, data: dict) -> None:
        # hint: Character(**data) lub z walidacja
        ...

reg = ExtendedCharacterRegistry()
reg.register_from_dict('paladin', {
    'name': 'PaladinTemplate',
    'char_class': 'paladin',
    'hp': 90,
    'mp': 60,
    'stats': {'str': 16, 'cha': 18},
    'inventory': ['holy sword', 'shield']
})
paladin = reg.create('paladin', 'Galahad')
print(paladin)

## 5. 🔹 Zastosowania i kiedy uzywac

Kiedy uzywac wzorca Prototype:
- Tworzenie obiektu jest drogie (ladowanie z bazy, skomplikowane obliczenia)
- Chcemy tworzyc wiele wariantow podobnego obiektu
- Klasa obiektu jest nieznana w czasie kompilacji
- Nie chcemy hierarchii klas fabrycznych

Kiedy NIE uzywac:
- Obiekty sa proste i tanie w tworzeniu
- Kopiowanie jest skomplikowane (cykliczne referencje, otwarte pliki)

| Wzorzec | Tworzy | Sposob |
|---|---|---|
| Factory | nowe obiekty | konstruktor |
| Builder | zlozone obiekty | krok po kroku |
| Prototype | kopie obiektow | klonowanie |

> 💡 W Pythonie `copy.deepcopy` jest universalnym mechanizmem
> klonowania. Implementuj `__deepcopy__` tylko gdy domyslne
> zachowanie jest nieprawidlowe.

In [ ]:
import copy, time
from dataclasses import dataclass, field

# Przyklad: konfiguracja serwera - tworzymy wiele wariantow
@dataclass
class ServerConfig:
    host: str
    port: int
    protocol: str
    headers: dict = field(default_factory=dict)
    middleware: list[str] = field(default_factory=list)
    timeout: int = 30

    def clone(self) -> 'ServerConfig':
        return copy.deepcopy(self)

# Bazowy szablon
base_config = ServerConfig(
    host='0.0.0.0',
    port=8080,
    protocol='http',
    headers={'Content-Type': 'application/json'},
    middleware=['auth', 'logging'],
)

# Warianty przez klonowanie
api_v1 = base_config.clone()
api_v1.port = 8081

api_v2 = base_config.clone()
api_v2.port = 8082
api_v2.protocol = 'https'
api_v2.middleware.append('rate_limit')

admin = base_config.clone()
admin.port = 9090
admin.middleware.append('admin_auth')

for cfg in [base_config, api_v1, api_v2, admin]:
    print(f'Port {cfg.port}: {cfg.protocol}, middleware={cfg.middleware}')

---

### 🐍 Cwiczenia - zastosowania

1. Napisz `EmailTemplate` z polami `subject`, `body`, `sender`,
   `recipients`. Stworz szablony 'welcome', 'newsletter', 'invoice'
   i klonuj je dla konkretnych odbiorcow.
2. Napisz `TestDataFactory` uzywajaca prototypow do generowania
   danych testowych: wiele kopii domyslnego uzytkownika z roznym id.
3. *(Trudniejsze)* Napisz `ConfigManager` z metodami `fork(name)`
   tworzaca nowy profil przez gleboka kopie biezacej konfiguracji
   i `switch(name)` przelaczajaca aktywny profil.

In [ ]:
# Cwiczenie 1: EmailTemplate
import copy
from dataclasses import dataclass, field

@dataclass
class EmailTemplate:
    subject: str
    body: str
    sender: str = 'noreply@example.com'
    recipients: list[str] = field(default_factory=list)

    def clone(self) -> 'EmailTemplate':
        ...

templates = {
    'welcome': EmailTemplate('Welcome!', 'Dear {name}, welcome!'),
    'newsletter': EmailTemplate('Newsletter #{n}', 'This week in tech...'),
}

welcome = copy.deepcopy(templates['welcome'])
welcome.recipients = ['alice@x.com']
print(f'welcome: {welcome}')

In [ ]:
# Cwiczenie 2: TestDataFactory
import copy
from dataclasses import dataclass

@dataclass
class User:
    id: int
    name: str
    email: str
    role: str = 'user'
    active: bool = True

default_user = User(id=0, name='TestUser', email='test@x.com')

def make_test_users(count: int) -> list[User]:
    users = []
    for i in range(1, count + 1):
        u = copy.deepcopy(default_user)
        u.id = i
        u.name = f'User{i}'
        u.email = f'user{i}@test.com'
        users.append(u)
    return users

users = make_test_users(5)
for u in users:
    print(u)

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: ConfigManager z profilami
import copy

class ConfigManager:
    def __init__(self, initial: dict):
        self._profiles: dict[str, dict] = {'default': copy.deepcopy(initial)}
        self._active = 'default'

    def fork(self, name: str) -> None:
        # hint: gleboka kopia biezacego profilu
        ...

    def switch(self, name: str) -> None:
        ...

    @property
    def config(self) -> dict:
        return self._profiles[self._active]

    def set(self, key: str, value) -> None:
        self.config[key] = value

mgr = ConfigManager({'debug': False, 'port': 8080, 'db': 'sqlite:///app.db'})
mgr.fork('production')
mgr.switch('production')
mgr.set('debug', False)
mgr.set('db', 'postgres://prod:5432/app')

mgr.switch('default')
print(f'default port: {mgr.config["port"]}')
print(f'default db: {mgr.config["db"]}')

mgr.switch('production')
print(f'production db: {mgr.config["db"]}')